# Dataset 2 — v1 Embeddings (Link Prediction, Log Target)

In [1]:
import sys
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.models.embeddings import GNNConfig, Node2VecConfig, extract_embeddings

EMB_ROOT = PROJECT_ROOT / 'src' / 'data' / 'embeddings'
pd.set_option('display.max_columns', 200)
print(f'Project root: {PROJECT_ROOT}')

def emb_path(name):
    name = str(name)
    ds  = 'dataset_3' if 'dataset3' in name else ('dataset_2' if 'dataset2' in name else 'dataset_1')
    sub = 'network_based' if name.startswith('node2vec') else 'feature_based'
    return EMB_ROOT / ds / sub / name

DATASET2_PATH = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_2'

def load_dataset2_nodes(dataset_path):
    nodes = pd.read_csv(dataset_path / 'nodes.csv').reset_index(drop=True)
    nodes['index'] = nodes.index
    nodes['Equity'] = nodes['buffer']; nodes['Assets'] = nodes['assets']
    return nodes

def load_dataset2_edges(dataset_path, bank_to_idx):
    matrix = pd.read_excel(dataset_path / 'network.xlsx', index_col=0)
    el = matrix.stack().reset_index(); el.columns = ['source_bank', 'target_bank', 'Weights']
    el = el[el['Weights'] != 0].copy()
    el['Sourceid'] = el['source_bank'].map(bank_to_idx); el['Targetid'] = el['target_bank'].map(bank_to_idx)
    el = el.dropna(subset=['Sourceid', 'Targetid'])
    el['Sourceid'] = el['Sourceid'].astype(int); el['Targetid'] = el['Targetid'].astype(int)
    return el[['Sourceid', 'Targetid', 'Weights']].reset_index(drop=True)

nodes = load_dataset2_nodes(DATASET2_PATH)
edges = load_dataset2_edges(DATASET2_PATH, dict(zip(nodes['bank'], nodes['index'])))
FEATURE_COLS = ['assets', 'liabilities', 'buffer']
target = pd.read_csv(DATASET2_PATH / 'targets' / 'target.csv')
target_cols = ['log_systemic_risk_label']   # log-only, matching Dataset 1

def build_dataset(config, output_path):
    emb, _ = extract_embeddings(edges, nodes, config, feature_cols=FEATURE_COLS)
    emb_cols = [c for c in emb.columns if c.startswith('emb_')]
    merged = emb[['bank_id'] + emb_cols].merge(target, on='bank_id', how='inner')[['bank_id'] + emb_cols + target_cols]
    merged.to_parquet(output_path, index=False)
    return merged


Project root: C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


In [2]:
TARGET_COL = 'log_systemic_risk_label'

cfg_graphsage_v1_32  = GNNConfig(hidden_dims=(256, 32),  dropout=0.3, lr=0.01, epochs=100, aggregation='mean', device='cpu')
cfg_graphsage_v1_64  = GNNConfig(hidden_dims=(256, 64),  dropout=0.3, lr=0.01, epochs=100, aggregation='mean', device='cpu')
cfg_graphsage_v1_128 = GNNConfig(hidden_dims=(256, 128), dropout=0.3, lr=0.01, epochs=100, aggregation='mean', device='cpu')

cfg_node2vec_v1_32   = Node2VecConfig(embedding_dim=32,  walk_length=20, context_size=10, walks_per_node=10, num_negative_samples=1, batch_size=128, lr=0.01, epochs=100, device='cpu')
cfg_node2vec_v1_64   = Node2VecConfig(embedding_dim=64,  walk_length=20, context_size=10, walks_per_node=10, num_negative_samples=1, batch_size=128, lr=0.01, epochs=100, device='cpu')
cfg_node2vec_v1_128  = Node2VecConfig(embedding_dim=128, walk_length=20, context_size=10, walks_per_node=10, num_negative_samples=5, batch_size=128, lr=0.01, epochs=100, device='cpu')

OUTPUTS = {
    'graphsage_v1_32':  emb_path('graphsage_v1_32_dataset2_dataset.parquet'),
    'graphsage_v1_64':  emb_path('graphsage_v1_64_dataset2_dataset.parquet'),
    'graphsage_v1_128': emb_path('graphsage_v1_128_dataset2_dataset.parquet'),
    'node2vec_v1_32':   emb_path('node2vec_v1_32_dataset2_dataset.parquet'),
    'node2vec_v1_64':   emb_path('node2vec_v1_64_dataset2_dataset.parquet'),
    'node2vec_v1_128':  emb_path('node2vec_v1_128_dataset2_dataset.parquet'),
}
OUTPUTS

{'graphsage_v1_32': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/dataset_2/feature_based/graphsage_v1_32_dataset2_dataset.parquet'),
 'graphsage_v1_64': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/dataset_2/feature_based/graphsage_v1_64_dataset2_dataset.parquet'),
 'graphsage_v1_128': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/dataset_2/feature_based/graphsage_v1_128_dataset2_dataset.parquet'),
 'node2vec_v1_32': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/dataset_2/network_based/node2vec_v1_32_dataset2_dataset.parquet'),
 'node2vec_v1_64': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/dataset_2/network_based/node2vec_v1_64_dataset2_dataset.parquet'),
 'node2vec_v1_128': WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/da

## GraphSAGE v1

In [3]:
for cfg, key in [
    (cfg_graphsage_v1_32,  'graphsage_v1_32'),
    (cfg_graphsage_v1_64,  'graphsage_v1_64'),
    (cfg_graphsage_v1_128, 'graphsage_v1_128'),
]:
    df = build_dataset(cfg, OUTPUTS[key])
    emb_cols = [c for c in df.columns if c.startswith('emb_')]
    print(f'{key:24s}  shape={df.shape}  emb_cols={len(emb_cols)}')

graphsage_v1_32           shape=(1444, 34)  emb_cols=32
graphsage_v1_64           shape=(1444, 66)  emb_cols=64
graphsage_v1_128          shape=(1444, 130)  emb_cols=128


## Node2Vec v1

In [4]:
for cfg, key in [
    (cfg_node2vec_v1_32,  'node2vec_v1_32'),
    (cfg_node2vec_v1_64,  'node2vec_v1_64'),
    (cfg_node2vec_v1_128, 'node2vec_v1_128'),
]:
    df = build_dataset(cfg, OUTPUTS[key])
    emb_cols = [c for c in df.columns if c.startswith('emb_')]
    print(f'{key:24s}  shape={df.shape}  emb_cols={len(emb_cols)}')

node2vec_v1_32            shape=(1444, 34)  emb_cols=32
node2vec_v1_64            shape=(1444, 66)  emb_cols=64
node2vec_v1_128           shape=(1444, 130)  emb_cols=128


## Output

In [5]:
for name, path in OUTPUTS.items():
    print(f'{name:24s} -> {path.name}')

graphsage_v1_32          -> graphsage_v1_32_dataset2_dataset.parquet
graphsage_v1_64          -> graphsage_v1_64_dataset2_dataset.parquet
graphsage_v1_128         -> graphsage_v1_128_dataset2_dataset.parquet
node2vec_v1_32           -> node2vec_v1_32_dataset2_dataset.parquet
node2vec_v1_64           -> node2vec_v1_64_dataset2_dataset.parquet
node2vec_v1_128          -> node2vec_v1_128_dataset2_dataset.parquet
